In [2]:
import os
import subprocess
from rdkit import Chem
from rdkit.Chem import rdFMCS
from rdkit.Chem import rdchem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

# This script processes all .mol2 files in a specified directory using the cgenff tool

mol2_dir = "/home/raheelx/cphmd_walkthrough/mol2_Epik"
output_dir = "/home/raheelx/cphmd_walkthrough/cgenff_output"
os.makedirs(output_dir, exist_ok=True)

for filename in sorted(os.listdir(mol2_dir)):
    if filename.endswith(".mol2"):
        mol2_path = os.path.join(mol2_dir, filename)
        basename = filename.replace(".mol2", "")
        output_str = os.path.join(output_dir, f"{basename}.str")
        cmd = f"module load cgenff && cgenff -a < {mol2_path} > {output_str}"
        try:
            subprocess.run(cmd, shell=True, executable="/bin/bash", check=True)
            print(f"Processed {filename}")
        except subprocess.CalledProcessError as e:
            print(f"Failed processing {filename}: {e}")



CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_1.mol2
Processed riboflavin_2.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_3.mol2
Processed riboflavin_4.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.



Processed riboflavin_5.mol2
Processed riboflavin_6.mol2


Now processing molecule Riboflav ...


In [12]:
# MCS + Clustering

# Convert mol2 files to sdf format using Open Babel
sdf_dir = "/home/raheelx/cphmd_walkthrough/sdf_from_cgenff"
os.makedirs(sdf_dir, exist_ok=True)

def convert_mol2_to_sdf(mol2_dir, sdf_dir):
    for filename in sorted(os.listdir(mol2_dir)):
        if filename.endswith(".mol2"):
            mol2_path = os.path.join(mol2_dir, filename)
            sdf_path = os.path.join(sdf_dir, filename.replace(".mol2",".sdf"))
            subprocess.run(["obabel", mol2_path, "-O", sdf_path], check = True)

# Load RDKit molecules

def load_sdfs(sdf_dir):
    mols = []
    for filename in sorted(os.listdir(sdf_dir)):
        if filename.endswith(".sdf"):
            path = os.path.join(sdf_dir, filename)
            mol = Chem.MolFromMolFile(path, sanitize=True)
            if mol:
                mols.append((filename.replace(".sdf", ""), mol))
            else:
                print(f"Failed to load {filename}")
    return mols

# Compute MCS

def compute_mcs(rdkit_mols):
     return rdFMCS.FindMCS(rdkit_mols,
                          timeout=60,
                          completeRingsOnly=True,
                          ringMatchesRingOnly=True,
                          threshold=0.9,
                          matchValences=True)

# Identify variable atoms (outside of MCS) and group into sites

def identify_variable_sites(mols, mcs_smarts):
    common_core = Chem.MolFromSmarts(mcs_smarts)
    for name, mol in mols:
        match = mol.GetSubstructMatch(common_core)  # <-- FIX here: use GetSubstructMatch (singular)
        if not match:
            print(f"No MCS match for {name}")
            continue
        core_atoms = set(match)
        variable_atoms = [a.GetIdx() for a in mol.GetAtoms() if a.GetIdx() not in core_atoms]

        # Group into connected components (sites)
        sites = []
        seen = set()
        for idx in variable_atoms:
            if idx in seen:
                continue
            site = set()
            stack = [idx]
            while stack:
                current = stack.pop()
                if current in seen:
                    continue
                seen.add(current)
                site.add(current)
                atom = mol.GetAtomWithIdx(current)
                for nbr in atom.GetNeighbors():
                    n_idx = nbr.GetIdx()
                    if n_idx in variable_atoms and n_idx not in seen:
                        stack.append(n_idx)
            sites.append(sorted(site))

        print(f"\nMolecule: {name}")
        print(f"Common core atom indices (MCS match): {sorted(core_atoms)}")
        print(f"Number of variable sites: {len(sites)}")
        for i, site in enumerate(sites):
            print(f" Site {i}: Atom indices {site}")
            
# === Execute Pipeline ===
convert_mol2_to_sdf(mol2_dir, sdf_dir)
mols = load_sdfs(sdf_dir)
print(f"\nLoaded {len(mols)} molecules")

rdkit_mols = [mol for name, mol in mols]
mcs_result = compute_mcs(rdkit_mols)
print("\nMCS SMARTS:", mcs_result.smartsString)
mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)
identify_variable_sites(mols, mcs_result.smartsString)

1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted
1 molecule converted



Loaded 6 molecules

MCS SMARTS: [#8&!R]-&!@[#6&!R](-&!@[#6&!R]-&!@[#6&!R]-&!@[#7&R])-&!@[#6&!R](-&!@[#8&!R])-&!@[#6&!R]-&!@[#8&!R]

Molecule: riboflavin_1
Common core atom indices (MCS match): [1, 2, 3, 6, 10, 11, 12, 13, 19]
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_2
Common core atom indices (MCS match): [1, 2, 3, 6, 10, 11, 12, 13, 19]
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_3
Common core atom indices (MCS match): [1, 2, 3, 6, 10, 11, 12, 13, 19]
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom indices [4, 5, 7, 8, 9, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26]

Molecule: riboflavin_4
Common core atom indices (MCS match): [1, 2, 3, 6, 10, 11, 12, 13, 19]
Number of variable sites: 2
 Site 0: Atom indices [0]
 Site 1: Atom 

*** Open Babel Warning  in ReadMolecule
  Failed to kekulize aromatic bonds in MOL2 file (title is Riboflavin)

1 molecule converted


In [14]:
# Drawing MCS output for visualization and check

viz_dir = "/home/raheelx/cphmd_walkthrough/mcs_visualizations"
os.makedirs(viz_dir, exist_ok=True)

for name, mol in mols:
    match = mol.GetSubstructMatch(mcs_mol)
    if match:
        AllChem.Compute2DCoords(mol)

        drawer = rdMolDraw2D.MolDraw2DCairo(400, 400)
        draw_opts = drawer.drawOptions()
        draw_opts.addAtomIndices = True  # This adds atom indices to the image
        drawer.DrawMolecule(mol, highlightAtoms=match, legend=name)
        drawer.FinishDrawing()

        img_path = os.path.join(viz_dir, f"{name}_highlighted_with_indices.png")
        with open(img_path, "wb") as f:
            f.write(drawer.GetDrawingText())
        print(f"Saved with atom indices: {img_path}")
    else:
        print(f"No MCS match found for {name}")




Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_1_highlighted_with_indices.png
Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_2_highlighted_with_indices.png
Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_3_highlighted_with_indices.png
Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_4_highlighted_with_indices.png
Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_5_highlighted_with_indices.png
Saved with atom indices: /home/raheelx/cphmd_walkthrough/mcs_visualizations/riboflavin_6_highlighted_with_indices.png
